In [1]:
# Preparacion

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

from src.data import cargar_enemdu
from src.features import agregar_variables

df = agregar_variables(cargar_enemdu())

In [2]:
# Construir la muestra

base = df[df["condact"].between(1, 6) & (df["ingrl"] > 0)]
print("Ocupados con ingreso > 0:", len(base))                       # debe dar 35302

en_edad = base["p03"].between(18, 65)
print("Fuera de 18-65 años:", (~en_edad).sum())

con_horas_ok = base["horas_total"] <= 84
print("Con más de 84 horas:", (~con_horas_ok).sum())

muestra = base[en_edad & con_horas_ok].copy()
print("Muestra final:", len(muestra))

muestra["ln_ingreso"] = np.log(muestra["ingrl"])
muestra["ln_horas"] = np.log(muestra["horas_total"])
muestra["mujer"] = (muestra["sexo"] == "Mujer").astype(int)
muestra["rural"] = (muestra["zona"] == "Rural").astype(int)
muestra["exper2"] = muestra["exper"] ** 2 / 100        # escalado para que el coeficiente sea legible
muestra["upm_id"] = pd.factorize(muestra["upm"])[0]

usadas = ["ln_ingreso", "anios_estudio", "exper", "exper2", "mujer", "rural",
          "ln_horas", "rama1", "nnivins", "fexp", "upm_id"]
print("\nNulos en las variables del modelo:")
print(muestra[usadas].isna().sum())

print("\nPercentiles del ingreso:")
print(muestra["ingrl"].describe(percentiles=[.01, .05, .5, .95, .99]).round(1))

Ocupados con ingreso > 0: 35302
Fuera de 18-65 años: 3019
Con más de 84 horas: 25
Muestra final: 32258

Nulos en las variables del modelo:
ln_ingreso       0
anios_estudio    0
exper            0
exper2           0
mujer            0
rural            0
ln_horas         0
rama1            0
nnivins          0
fexp             0
upm_id           0
dtype: int64

Percentiles del ingreso:
count    32258.0
mean       603.2
std        604.5
min          1.0
1%          25.0
5%          80.0
50%        482.0
95%       1600.0
99%       3000.0
max      18310.0
Name: ingrl, dtype: float64


In [3]:
# Estimar los 5 modelos

def ajustar(formula, d):
    return smf.wls(formula, data=d, weights=d["fexp"]).fit(
        cov_type="cluster", cov_kwds={"groups": d["upm_id"]})

f0 = "ln_ingreso ~ mujer"
f1 = "ln_ingreso ~ anios_estudio + exper + exper2"
f2 = f1 + " + mujer + rural"
f3 = f2 + " + ln_horas + C(rama1)"
f4 = "ln_ingreso ~ C(nnivins) + exper + exper2 + mujer + rural + ln_horas + C(rama1)"

modelos = {"M0": ajustar(f0, muestra), "M1": ajustar(f1, muestra), "M2": ajustar(f2, muestra),
           "M3": ajustar(f3, muestra), "M4": ajustar(f4, muestra)}

claves = ["mujer", "anios_estudio", "exper", "exper2", "rural", "ln_horas"]
coef = pd.DataFrame({k: {v: r.params.get(v, np.nan) for v in claves} for k, r in modelos.items()})
err = pd.DataFrame({k: {v: r.bse.get(v, np.nan) for v in claves} for k, r in modelos.items()})
print("Coeficientes:\n", coef.round(3), "\n")
print("Errores estándar (agrupados por UPM):\n", err.round(3), "\n")
print({k: (round(r.rsquared, 3), int(r.nobs)) for k, r in modelos.items()})

Coeficientes:
                   M0     M1     M2     M3     M4
mujer         -0.242    NaN -0.351 -0.175 -0.186
anios_estudio    NaN  0.090  0.086  0.043    NaN
exper            NaN  0.032  0.031  0.014  0.015
exper2           NaN -0.048 -0.046 -0.019 -0.025
rural            NaN    NaN -0.333 -0.224 -0.219
ln_horas         NaN    NaN    NaN  1.101  1.106 

Errores estándar (agrupados por UPM):
                   M0     M1     M2     M3     M4
mujer          0.042    NaN  0.034  0.025  0.024
anios_estudio    NaN  0.004  0.003  0.003    NaN
exper            NaN  0.004  0.004  0.003  0.003
exper2           NaN  0.008  0.008  0.008  0.007
rural            NaN    NaN  0.067  0.053  0.048
ln_horas         NaN    NaN    NaN  0.027  0.027 

{'M0': (np.float64(0.016), 32258), 'M1': (np.float64(0.167), 32258), 'M2': (np.float64(0.223), 32258), 'M3': (np.float64(0.524), 32258), 'M4': (np.float64(0.529), 32258)}


In [4]:
# Interpretar

def pct(b):
    return 100 * (np.exp(b) - 1)

r2, r3, r0 = modelos["M2"], modelos["M3"], modelos["M0"]

print(f"Brecha bruta (M0): {pct(r0.params['mujer']):.1f}%")
print(f"Brecha ajustada por educación, experiencia y zona (M2): {pct(r2.params['mujer']):.1f}%")
print(f"Brecha ajustada además por horas y rama (M3): {pct(r3.params['mujer']):.1f}%")
print(f"\nRetorno por año de estudio (M1): {pct(modelos['M1'].params['anios_estudio']):.1f}%")
print(f"Retorno por año de estudio (M3): {pct(r3.params['anios_estudio']):.1f}%")
print(f"Diferencia rural vs. urbano (M2): {pct(r2.params['rural']):.1f}%")

b1, b2 = r2.params["exper"], r2.params["exper2"]
print(f"\nExperiencia a la que el ingreso es máximo (M2): {-b1 / (2 * b2 / 100):.1f} años")

Brecha bruta (M0): -21.5%
Brecha ajustada por educación, experiencia y zona (M2): -29.6%
Brecha ajustada además por horas y rama (M3): -16.1%

Retorno por año de estudio (M1): 9.4%
Retorno por año de estudio (M3): 4.4%
Diferencia rural vs. urbano (M2): -28.3%

Experiencia a la que el ingreso es máximo (M2): 33.7 años


In [5]:
perfil = muestra.groupby("sexo").apply(lambda s: pd.Series({
    "años de estudio": np.average(s["anios_estudio"], weights=s["fexp"]),
    "experiencia potencial": np.average(s["exper"], weights=s["fexp"]),
    "% rural": 100 * np.average(s["rural"], weights=s["fexp"]),
    "horas semanales": np.average(s["horas_total"], weights=s["fexp"]),
})).round(1)
print(perfil)

        años de estudio  experiencia potencial  % rural  horas semanales
sexo                                                                    
Hombre             10.8                   23.3     29.7             39.2
Mujer              11.8                   22.7     23.0             33.8


In [6]:
w = muestra["fexp"]
print("% con ingreso exactamente 482:", round(100 * w[muestra["ingrl"] == 482].sum() / w.sum(), 1))

% con ingreso exactamente 482: 2.5


In [7]:
def pct(b):
    return 100 * (np.exp(b) - 1)

p01, p99 = muestra["ingrl"].quantile([0.01, 0.99])
muestras = {
    "completa": muestra,
    "sin 1% extremos de ingreso": muestra[muestra["ingrl"].between(p01, p99)],
    "tiempo completo (35-48 h)": muestra[muestra["horas_total"].between(35, 48)],
}

filas = []
for nombre, d in muestras.items():
    for etiqueta, f in [("M2", f2), ("M3", f3)]:
        r = ajustar(f, d)
        filas.append({"muestra": nombre, "modelo": etiqueta, "n": int(r.nobs),
                      "mujer %": round(pct(r.params["mujer"]), 1),
                      "estudio %": round(pct(r.params["anios_estudio"]), 1)})
print(pd.DataFrame(filas))

                      muestra modelo      n  mujer %  estudio %
0                    completa     M2  32258    -29.6        8.9
1                    completa     M3  32258    -16.1        4.4
2  sin 1% extremos de ingreso     M2  31690    -26.2        8.4
3  sin 1% extremos de ingreso     M3  31690    -13.6        4.1
4   tiempo completo (35-48 h)     M2  19720     -2.5        7.8
5   tiempo completo (35-48 h)     M3  19720     -7.4        5.0


In [8]:
b = modelos["M4"].params.filter(like="nnivins")
print(pd.DataFrame({"coef": b.round(3), "%": (100 * (np.exp(b) - 1)).round(1)}))

                    coef     %
C(nnivins)[T.2.0] -0.084  -8.1
C(nnivins)[T.3.0]  0.053   5.5
C(nnivins)[T.4.0]  0.195  21.5
C(nnivins)[T.5.0]  0.551  73.4


In [9]:
import os
os.makedirs("../reports", exist_ok=True)
pd.concat({"coef": coef, "error_estandar": err}, axis=1).round(4).to_csv(
    "../reports/mincer_coeficientes.csv", encoding="utf-8-sig")

In [10]:
# Aporte de las horas e Intervalos para M4

r_h = ajustar(f2 + " + ln_horas", muestra)
print("R² M2:", round(modelos["M2"].rsquared, 3), "| M2 + horas:", round(r_h.rsquared, 3),
      "| M3:", round(modelos["M3"].rsquared, 3))
print("Brecha M2 + horas:", round(pct(r_h.params["mujer"]), 1), "%")

b4 = modelos["M4"]
niv = b4.params.filter(like="nnivins").index
print(pd.DataFrame({
    "%": (100 * (np.exp(b4.params[niv]) - 1)).round(1),
    "IC95 inf": (100 * (np.exp(b4.params[niv] - 1.96 * b4.bse[niv]) - 1)).round(1),
    "IC95 sup": (100 * (np.exp(b4.params[niv] + 1.96 * b4.bse[niv]) - 1)).round(1)}))

R² M2: 0.223 | M2 + horas: 0.471 | M3: 0.524
Brecha M2 + horas: -8.7 %
                      %  IC95 inf  IC95 sup
C(nnivins)[T.2.0]  -8.1     -40.2      41.2
C(nnivins)[T.3.0]   5.5      -4.9      17.0
C(nnivins)[T.4.0]  21.5       9.0      35.4
C(nnivins)[T.5.0]  73.4      54.9      94.2


In [11]:
filas = []
for nombre, d in muestras.items():
    for etiqueta, f in [("M2", f2), ("M3", f3)]:
        r = ajustar(f, d)
        lo, hi = r.conf_int().loc["mujer"]
        filas.append({"muestra": nombre, "modelo": etiqueta,
                      "mujer %": round(pct(r.params["mujer"]), 1),
                      "IC95 inf": round(pct(lo), 1), "IC95 sup": round(pct(hi), 1)})
print(pd.DataFrame(filas))

print(modelos["M4"].t_test("C(nnivins)[T.5.0] - C(nnivins)[T.4.0] = 0"))

                      muestra modelo  mujer %  IC95 inf  IC95 sup
0                    completa     M2    -29.6     -34.1     -24.7
1                    completa     M3    -16.1     -20.1     -11.8
2  sin 1% extremos de ingreso     M2    -26.2     -31.1     -21.0
3  sin 1% extremos de ingreso     M3    -13.6     -17.7      -9.3
4   tiempo completo (35-48 h)     M2     -2.5      -9.8       5.3
5   tiempo completo (35-48 h)     M3     -7.4     -12.6      -2.0
                             Test for Constraints                             
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
c0             0.3558      0.021     17.307      0.000       0.315       0.396


In [12]:
r_h = ajustar(f2 + " + ln_horas", muestra)
lo, hi = r_h.conf_int().loc["mujer"]
print(f"M2 + horas: {pct(r_h.params['mujer']):.1f}%  (IC 95%: {pct(lo):.1f} a {pct(hi):.1f})")

pago = muestra.groupby("rama1").apply(lambda s: np.average(s["ln_ingreso"], weights=s["fexp"]))
muestra["pago_rama"] = muestra["rama1"].map(pago)
print(muestra.groupby("sexo").apply(lambda s: np.average(s["pago_rama"], weights=s["fexp"])).round(3))

M2 + horas: -8.7%  (IC 95%: -13.8 a -3.4)
sexo
Hombre    5.850
Mujer     5.948
dtype: float64


In [13]:
pd.DataFrame(filas).to_csv("../reports/brecha_mujer.csv", index=False, encoding="utf-8-sig")